In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

print("Libraries imported successfully!")

Libraries imported successfully!


In [6]:
file_path = "/content/online_retail_II.xlsx"

excel_data = pd.read_excel(
    file_path,
    sheet_name=None
)

print("Worksheets:", list(excel_data.keys()))

for name, data in excel_data.items():
    print(name, data.shape)

Worksheets: ['Year 2009-2010', 'Year 2010-2011']
Year 2009-2010 (525461, 8)
Year 2010-2011 (541910, 8)


In [7]:
df = pd.concat(
    excel_data.values(),
    ignore_index=True
)

df_clean = df.copy()

print("Dataset loaded successfully!")
print("Dataset shape:", df_clean.shape)

Dataset loaded successfully!
Dataset shape: (1067371, 8)


In [8]:
display(df_clean.head())
print("Columns:", df_clean.columns.tolist())
print("\nDataset information:")
df_clean.info()
display(df_clean.describe())
display(df_clean.describe(include="object"))
print("Unique invoices:", df_clean["Invoice"].nunique())
print("Unique products:", df_clean["StockCode"].nunique())
print("Unique customers:", df_clean["Customer ID"].nunique())
print("Unique countries:", df_clean["Country"].nunique())

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 65.1+ MB


,Quantity,InvoiceDate,Price,Customer ID
count,1.067371e+06,1067371,1.067371e+06,824364.000000
mean,9.938898e+00,2011-01-02 21:13:55.394028544,4.649388e+00,15324.638504
min,-8.099500e+04,2009-12-01 07:45:00,-5.359436e+04,12346.000000
25%,1.000000e+00,2010-07-09 09:46:00,1.250000e+00,13975.000000
50%,3.000000e+00,2010-12-07 15:28:00,2.100000e+00,15255.000000
75%,1.000000e+01,2011-07-22 10:23:00,4.150000e+00,16797.000000
max,8.099500e+04,2011-12-09 12:50:00,3.897000e+04,18287.000000
std,1.727058e+02,NaN,1.235531e+02,1697.464450


,Invoice,StockCode,Description,Country
count,1067371,1067371,1062989,1067371
unique,53628,5305,5698,43
top,537434,85123A,WHITE HANGING HEART T-LIGHT HOLDER,United Kingdom
freq,1350,5829,5918,981330


Unique invoices: 53628
Unique products: 5305
Unique customers: 5942
Unique countries: 43


In [9]:
df_clean.columns = df_clean.columns.str.strip()

df_clean.rename(
    columns={
        "Customer ID": "CustomerID",
        "Price": "UnitPrice"
    },
    inplace=True
)

df_clean["InvoiceDate"] = pd.to_datetime(
    df_clean["InvoiceDate"],
    errors="coerce"
)

df_clean["CustomerID"] = df_clean["CustomerID"].astype("string")

df_clean["Quantity"] = pd.to_numeric(
    df_clean["Quantity"],
    errors="coerce"
)

df_clean["UnitPrice"] = pd.to_numeric(
    df_clean["UnitPrice"],
    errors="coerce"
)

print(df_clean.dtypes)

Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID     string[python]
Country                object
dtype: object


In [10]:
missing_report = pd.DataFrame({
    "Missing Count": df_clean.isnull().sum(),
    "Missing Percentage": (
        df_clean.isnull().sum() / len(df_clean)
    ) * 100
})

display(missing_report.sort_values(
    by="Missing Percentage",
    ascending=False
))

# Fill missing product descriptions
df_clean["Description"] = df_clean["Description"].fillna("Unknown")

# Remove records missing essential transaction information
df_clean.dropna(
    subset=["Invoice", "InvoiceDate", "Quantity", "UnitPrice"],
    inplace=True
)

print("Missing values handled successfully!")

,Missing Count,Missing Percentage
CustomerID,243007,22.766873
Description,4382,0.410541
StockCode,0,0.000000
Invoice,0,0.000000
Quantity,0,0.000000
InvoiceDate,0,0.000000
UnitPrice,0,0.000000
Country,0,0.000000


Missing values handled successfully!


In [11]:
print("Duplicate records:", df_clean.duplicated().sum())
before = len(df_clean)

df_clean.drop_duplicates(inplace=True)

after = len(df_clean)

print("Rows before:", before)
print("Rows after:", after)
print("Duplicates removed:", before - after)

Duplicate records: 34335
Rows before: 1067371
Rows after: 1033036
Duplicates removed: 34335


In [12]:
df_clean["Invoice"] = df_clean["Invoice"].astype("string")

df_clean["IsCancellation"] = (
    df_clean["Invoice"].str.startswith("C", na=False)
)

df_clean["IsNegativeQuantity"] = (
    df_clean["Quantity"] < 0
)
print("Cancellation records:",
      df_clean["IsCancellation"].sum())

print("Negative quantity records:",
      df_clean["IsNegativeQuantity"].sum())

display(
    df_clean[df_clean["IsCancellation"]].head()
)

Cancellation records: 19104
Negative quantity records: 22496


,Invoice,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,IsCancellation,IsNegativeQuantity
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,True,True
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia,True,True
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia,True,True
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia,True,True
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,True,True


In [13]:
purchase_transactions = df_clean[
    (~df_clean["IsCancellation"]) &
    (df_clean["Quantity"] > 0) &
    (df_clean["UnitPrice"] > 0) &
    (df_clean["CustomerID"].notna())
].copy()

print("Original cleaned dataset:", df_clean.shape)
print("Purchase dataset:", purchase_transactions.shape)

Original cleaned dataset: (1033036, 10)
Purchase dataset: (779425, 10)


In [14]:
purchase_transactions["Revenue"] = (
    purchase_transactions["Quantity"] *
    purchase_transactions["UnitPrice"]
)

display(purchase_transactions.head())

,Invoice,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,IsCancellation,IsNegativeQuantity,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,False,False,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,False,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,False,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,False,False,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,False,False,30.0


In [15]:
purchase_transactions["InvoiceYear"] = (
    purchase_transactions["InvoiceDate"].dt.year
)

purchase_transactions["InvoiceMonth"] = (
    purchase_transactions["InvoiceDate"].dt.month
)

purchase_transactions["InvoiceDay"] = (
    purchase_transactions["InvoiceDate"].dt.day
)

purchase_transactions["InvoiceHour"] = (
    purchase_transactions["InvoiceDate"].dt.hour
)

purchase_transactions["InvoiceDayOfWeek"] = (
    purchase_transactions["InvoiceDate"].dt.dayofweek
)

display(purchase_transactions.head())

,Invoice,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,IsCancellation,IsNegativeQuantity,Revenue,InvoiceYear,InvoiceMonth,InvoiceDay,InvoiceHour,InvoiceDayOfWeek
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,False,False,83.4,2009,12,1,7,1
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,False,81.0,2009,12,1,7,1
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,False,81.0,2009,12,1,7,1
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,False,False,100.8,2009,12,1,7,1
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,False,False,30.0,2009,12,1,7,1


In [16]:
print("FINAL DATASET VALIDATION")

print("\nDataset shape:", purchase_transactions.shape)

print("\nMissing values:")
print(purchase_transactions.isnull().sum())

print("\nDuplicate records:",
      purchase_transactions.duplicated().sum())

print("\nInvalid quantities:",
      (purchase_transactions["Quantity"] <= 0).sum())

print("\nInvalid unit prices:",
      (purchase_transactions["UnitPrice"] <= 0).sum())

print("\nInvalid revenue:",
      (purchase_transactions["Revenue"] <= 0).sum())

print("\nUnique customers:",
      purchase_transactions["CustomerID"].nunique())

print("\nUnique invoices:",
      purchase_transactions["Invoice"].nunique())

print("\nDate range:")
print(purchase_transactions["InvoiceDate"].min())
print(purchase_transactions["InvoiceDate"].max())


FINAL DATASET VALIDATION

Dataset shape: (779425, 16)

Missing values:
Invoice               0
StockCode             0
Description           0
Quantity              0
InvoiceDate           0
UnitPrice             0
CustomerID            0
Country               0
IsCancellation        0
IsNegativeQuantity    0
Revenue               0
InvoiceYear           0
InvoiceMonth          0
InvoiceDay            0
InvoiceHour           0
InvoiceDayOfWeek      0
dtype: int64

Duplicate records: 0

Invalid quantities: 0

Invalid unit prices: 0

Invalid revenue: 0

Unique customers: 5878

Unique invoices: 36969

Date range:
2009-12-01 07:45:00
2011-12-09 12:50:00


In [17]:
import os

os.makedirs("/content/data/processed", exist_ok=True)

df_clean.to_csv(
    "/content/data/processed/cleaned_transactions.csv",
    index=False
)

purchase_transactions.to_csv(
    "/content/data/processed/purchase_transactions.csv",
    index=False
)

print("Datasets saved successfully!")

Datasets saved successfully!


In [18]:
from google.colab import files

files.download(
    "/content/data/processed/purchase_transactions.csv"
)

files.download(
    "/content/data/processed/cleaned_transactions.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>